In [1]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error,confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA
from scipy import stats
import sys
import seaborn as sns
import scipy.signal
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *
import TA_tools
import time
import importlib
importlib.reload(TA_tools)

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/adversary/TA_segmented_initialise.ipynb"

INFO: Found ChipWhisperer😍


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:397) Could not adjust adc_mul via output divider alone. Recalcing clocks...
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:398) Target clock has dropped for a moment. You may need to reset your target


scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_tolerance             changed from 1144409.1796875           to 13096723.705530167       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [3]:
tools = TA_tools.Tools(scope,target)
N=8
seg_count = int((N/2)*np.log2(N))

In [4]:
sigma = [0]#,1]#,2,3,4,5,6,7,8,9,10]
repeats = 10
L = 3
rec_rate = []
times = []
for sig in sigma:
    results = []
    for r in range(repeats):
        cor_count, time_taken = tools.run_attack(L,sig,N,seg_count)
        results.append(cor_count)
        if cor_count == N:
            times.append(time_taken)

    rec_rate.append(np.asarray(results).mean(axis=0)/N)

[1 0 1 1 1 1 1 0]


Segment 0
Guess 0. MSE: 0.026930091417854338, PC: PearsonRResult(statistic=0.4978433287673936, pvalue=0.0)
Guess 1. MSE: 0.03299193977045687, PC: PearsonRResult(statistic=0.3856896788187195, pvalue=0.0)
Guess 2. MSE: 0.04104327327827073, PC: PearsonRResult(statistic=0.2338513997655182, pvalue=9.653636295534209e-149)
Guess 3. MSE: 0.00018392346275788736, PC: PearsonRResult(statistic=0.9965736472717602, pvalue=0.0)


Segment 1
Guess 0. MSE: 0.026820513880910418, PC: PearsonRResult(statistic=0.4988976383714928, pvalue=0.0)
Guess 1. MSE: 0.03287604392866953, PC: PearsonRResult(statistic=0.38630347239872675, pvalue=0.0)
Guess 2. MSE: 0.041030545161205814, PC: PearsonRResult(statistic=0.23421260885129563, pvalue=3.2965302010741043e-149)
Guess 3. MSE: 0.00018041203381007583, PC: PearsonRResult(statistic=0.99663756336392, pvalue=0.0)


Segment 2
Guess 0. MSE: 0.03980557767162247, PC: PearsonRResult(statistic=0.2560151951593098, pvalue=7.104451016745321e-179)
Guess 1. MSE: 0

In [ ]:
times = np.asarray(times)
np.savetxt("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/results/times_8_bit.csv",times,delimiter=",")

In [ ]:
print(times)

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
plt.plot(sigma,rec_rate)
plt.xlabel("Noise")
plt.ylabel("Recovery Rate")
plt.show()

Section where we aim to show the difference between each of the four hypothesis inputs.

In [ ]:
x1 = "0000000000000000\n"
x2 = "0000000010000000\n"
x3 = "1000000000000000\n"
x4 = "1000000010000000\n"

x1_avg = []
x2_avg = []
x3_avg = []
x4_avg = []

for _ in range(10):
    x1_avg.append(tools.get_trace(x1)[0])
    x2_avg.append(tools.get_trace(x2)[0])
    x3_avg.append(tools.get_trace(x3)[0])
    x4_avg.append(tools.get_trace(x4)[0])

x1_avg = np.asarray(x1_avg).mean(axis=0)
x2_avg = np.asarray(x2_avg).mean(axis=0)
x3_avg = np.asarray(x3_avg).mean(axis=0)
x4_avg = np.asarray(x4_avg).mean(axis=0)


tools.plot_overlay([x1_avg, x2_avg, x3, x3_avg, x4_avg])
